In [ ]:
!pip install fastf1

In [ ]:
import os
import pandas as pd
import fastf1

# ==========================
# CHANGE ONLY THIS
# ==========================
YEAR = 2018
# ==========================

CACHE_DIR = "cache"
OUTPUT_DIR = "yearly_data"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

fastf1.Cache.enable_cache(CACHE_DIR)

all_laps = []

schedule = fastf1.get_event_schedule(YEAR)

for _, event in schedule.iterrows():

    race = event["EventName"]

    print(f"Downloading {YEAR} - {race}")

    try:
        session = fastf1.get_session(YEAR, race, "R")
        session.load()

        laps = session.laps.copy()

        if laps.empty:
            continue

        # Metadata
        laps["Season"] = YEAR
        laps["Round"] = event["RoundNumber"]
        laps["Race"] = race

        # Convert times to seconds
        for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
            if col in laps.columns:
                laps[col] = laps[col].dt.total_seconds()

        columns = [
            "Season",
            "Round",
            "Race",
            "Driver",
            "Team",
            "LapNumber",
            "LapTime",
            "Sector1Time",
            "Sector2Time",
            "Sector3Time",
            "Compound",
            "TyreLife",
            "Stint",
            "Position",
            "TrackStatus"
        ]

        columns = [c for c in columns if c in laps.columns]

        all_laps.append(laps[columns])

    except Exception as e:
        print(f"Skipped {race}: {e}")

# Save one CSV for the year
if all_laps:
    final = pd.concat(all_laps, ignore_index=True)

    filename = os.path.join(OUTPUT_DIR, f"F1_{YEAR}.csv")
    final.to_csv(filename, index=False)

    print(f"\nSaved: {filename}")
    print(f"Rows: {len(final):,}")
else:
    print("No data downloaded.")